# Prepared Table Filter Example

This notebook shows how to:

- read a canonical prepared-table bundle from disk
- apply a household-level filter
- propagate that filter to related prepared tables
- write a new prepared-table bundle to a different directory
- generate a `prepared_table_map` snippet you can paste into config

It is designed as a realistic test for the `prepared_table_map` feature.

## Assumptions

- Input files are canonical prepared tables, typically from a normal prepared-cache directory like `<summary_root>/<run_key>/prepared_tables/`
- Table names are:
  - `households`
  - `persons`
  - `tours`
  - `trips`
  - `joint_tour_participants`
  - `land_use`
- Files may be either `.parquet` or `.csv`
- `land_use` is not filtered because it is zone-level lookup data, not household/person/tour/trip data

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import polars as pl

In [2]:
# Update these paths before running.
# INPUT_DIR should usually be a prepared-cache directory that already contains
# canonical prepared tables, for example:
#   artifacts/summary_cache/base/prepared_tables

INPUT_DIR = Path(r"configs/artifacts/new-estimation-output/prepared_tables")
OUTPUT_DIR = Path(r"example_output_files/base_households_filtered")

# Output format for the rewritten prepared tables.
# Supported values: "parquet" or "csv"
OUTPUT_FORMAT = "csv"

# Canonical prepared table ids used by the visualizer.
TABLE_IDS = [
    "households",
    "persons",
    "tours",
    "trips",
    "joint_tour_participants",
    "land_use",
]

assert OUTPUT_FORMAT in {"parquet", "csv"}

In [3]:
def find_table_file(root: Path, table_id: str) -> Path:
    for suffix in (".parquet", ".csv"):
        candidate = root / f"{table_id}{suffix}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {table_id}.parquet or {table_id}.csv in {root}")


def read_table(path: Path) -> pl.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pl.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pl.read_csv(path, infer_schema_length=10000)
    raise ValueError(f"Unsupported file type: {path}")


def write_table(df: pl.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        df.write_parquet(path)
        return
    if path.suffix.lower() == ".csv":
        df.write_csv(path)
        return
    raise ValueError(f"Unsupported output type: {path}")


def output_path(output_dir: Path, table_id: str, output_format: str) -> Path:
    return output_dir / f"{table_id}.{output_format}"


def normalize_numeric_string_column(df: pl.DataFrame, column: str) -> pl.DataFrame:
    if column not in df.columns:
        return df
    if df.schema[column] != pl.String:
        return df
    return df.with_columns(
        pl.col(column)
        .str.replace_all(",", "")
        .cast(pl.Float64, strict=False)
        .alias(column)
    )


def normalize_common_numeric_string_columns(table_id: str, df: pl.DataFrame) -> pl.DataFrame:
    candidates_by_table = {
        "households": ["finalweight", "sample_rate", "auto_ownership", "hhsize", "HHSIZE"],
        "persons": ["finalweight"],
        "tours": ["finalweight", "SKIMDIST", "start_hour", "end_hour", "tourdur", "AWDT"],
        "trips": ["finalweight", "od_dist", "out_dir_dist", "depart_hour", "stops", "AWDT"],
        "joint_tour_participants": [],
        "land_use": ["employment_count", "enrollment_count", "TAZ", "MAZ", "zone_id"],
    }
    result = df
    for column in candidates_by_table.get(table_id, []):
        result = normalize_numeric_string_column(result, column)
    return result

In [4]:
input_files = {table_id: find_table_file(INPUT_DIR, table_id) for table_id in TABLE_IDS}
tables = {table_id: read_table(path) for table_id, path in input_files.items()}

for table_id, path in input_files.items():
    print(f"{table_id:25s} <- {path}")

households                <- configs\artifacts\new-estimation-output\prepared_tables\households.parquet
persons                   <- configs\artifacts\new-estimation-output\prepared_tables\persons.parquet
tours                     <- configs\artifacts\new-estimation-output\prepared_tables\tours.parquet
trips                     <- configs\artifacts\new-estimation-output\prepared_tables\trips.parquet
joint_tour_participants   <- configs\artifacts\new-estimation-output\prepared_tables\joint_tour_participants.parquet
land_use                  <- configs\artifacts\new-estimation-output\prepared_tables\land_use.parquet


## Choose a realistic household filter

This example tries to keep households with either:

- `HHSIZE >= 2` when `HHSIZE` exists, or
- `hhsize >= 2` when `hhsize` exists

If neither exists, it falls back to keeping every other household by row number.

Feel free to replace this with any custom condition you want.

In [5]:
households = tables["households"]

if "household_id" not in households.columns:
    raise ValueError("Prepared households table must contain household_id")

if "HHSIZE" in households.columns:
    filtered_households = households.filter(pl.col("HHSIZE") >= 2)
    household_filter_description = "HHSIZE >= 2"
elif "hhsize" in households.columns:
    filtered_households = households.filter(pl.col("hhsize") >= 2)
    household_filter_description = "hhsize >= 2"
else:
    filtered_households = households.with_row_index("_row_id").filter(
        (pl.col("_row_id") % 2) == 0
    ).drop("_row_id")
    household_filter_description = "fallback row-based filter: keep every other household"

if filtered_households.is_empty():
    raise ValueError("The chosen household filter removed every household. Adjust the filter before continuing.")

print("Household filter:", household_filter_description)
print("Original households:", households.height)
print("Filtered households:", filtered_households.height)

Household filter: HHSIZE >= 2
Original households: 43637
Filtered households: 26513


## Propagate the filter to related prepared tables

This is the key part for producing a consistent custom prepared bundle.

- `persons` is filtered by surviving `household_id`
- `tours` is filtered by surviving `household_id` and `person_id` when available
- `trips` is filtered by surviving `household_id`, `person_id`, and `tour_id` when available
- `joint_tour_participants` is filtered by surviving `person_id` and `tour_id` when available
- `land_use` is left unchanged

In [6]:
persons = tables["persons"]
tours = tables["tours"]
trips = tables["trips"]
joint_participants = tables["joint_tour_participants"]
land_use = tables["land_use"]

household_ids = filtered_households.select("household_id").unique()

filtered_persons = persons
if "household_id" in filtered_persons.columns:
    filtered_persons = filtered_persons.join(household_ids, on="household_id", how="inner")

person_ids = (
    filtered_persons.select("person_id").unique()
    if "person_id" in filtered_persons.columns
    else pl.DataFrame({"person_id": []}, schema={"person_id": pl.Int64})
)

filtered_tours = tours
if "household_id" in filtered_tours.columns:
    filtered_tours = filtered_tours.join(household_ids, on="household_id", how="inner")
if "person_id" in filtered_tours.columns and "person_id" in person_ids.columns:
    filtered_tours = filtered_tours.join(person_ids, on="person_id", how="inner")

tour_ids = (
    filtered_tours.select("tour_id").unique()
    if "tour_id" in filtered_tours.columns
    else pl.DataFrame({"tour_id": []}, schema={"tour_id": pl.Int64})
)

filtered_trips = trips
if "household_id" in filtered_trips.columns:
    filtered_trips = filtered_trips.join(household_ids, on="household_id", how="inner")
if "person_id" in filtered_trips.columns and "person_id" in person_ids.columns:
    filtered_trips = filtered_trips.join(person_ids, on="person_id", how="inner")
if "tour_id" in filtered_trips.columns and "tour_id" in tour_ids.columns:
    filtered_trips = filtered_trips.join(tour_ids, on="tour_id", how="inner")

filtered_joint_participants = joint_participants
if "person_id" in filtered_joint_participants.columns and "person_id" in person_ids.columns:
    filtered_joint_participants = filtered_joint_participants.join(person_ids, on="person_id", how="inner")
if "tour_id" in filtered_joint_participants.columns and "tour_id" in tour_ids.columns:
    filtered_joint_participants = filtered_joint_participants.join(tour_ids, on="tour_id", how="inner")

filtered_tables = {
    "households": filtered_households,
    "persons": filtered_persons,
    "tours": filtered_tours,
    "trips": filtered_trips,
    "joint_tour_participants": filtered_joint_participants,
    "land_use": land_use,
}

In [7]:
summary_rows = []
for table_id in TABLE_IDS:
    original = tables[table_id]
    filtered = filtered_tables[table_id]
    summary_rows.append(
        {
            "table": table_id,
            "original_rows": original.height,
            "filtered_rows": filtered.height,
            "rows_removed": original.height - filtered.height,
        }
    )

pl.DataFrame(summary_rows)

table,original_rows,filtered_rows,rows_removed
str,i64,i64,i64
"""households""",43637,26513,17124
"""persons""",86280,69156,17124
"""tours""",76704,59795,16909
"""trips""",218669,168795,49874
"""joint_tour_participants""",0,0,0
"""land_use""",66999,66999,0


## Normalize common numeric string columns before writing

This step is especially helpful for CSV output. It converts common numeric-looking prepared columns such as `enrollment_count` and `employment_count` back to numeric types when they were read as strings, including values like `"4,238"`.


In [8]:
filtered_tables = {
    table_id: normalize_common_numeric_string_columns(table_id, df)
    for table_id, df in filtered_tables.items()
}

for table_id in TABLE_IDS:
    print(table_id, filtered_tables[table_id].schema)

households Schema({'password': String, 'rm_household_id': Int64, 'ms_household_id': Int64, 'participation_group': Int64, 'incentive': Int64, 'incentive_amount': Int64, 'first_travel_date': String, 'last_travel_date': String, 'browser': Int64, 'tester': Int64, 'num_days_complete': Int64, 'is_complete': Int64, 'disposition': Int64, 'signup_complete_time': String, 'signup_platform': String, 'diary_platform': String, 'signup_rmove': Int64, 'signup_call_center': Int64, 'diary_call_center': Int64, 'num_days_complete_weekday': Int64, 'num_complete_mon': Int64, 'num_complete_tue': Int64, 'num_complete_wed': Int64, 'num_complete_thu': Int64, 'num_complete_fri': Int64, 'num_days_complete_weekend': Int64, 'num_complete_sat': Int64, 'num_complete_sun': Int64, 'num_trips': Int64, 'num_people_survey': Int64, 'num_people': Int64, 'raw_person_count': Int64, 'num_surveyable': Int64, 'num_activated': Int64, 'num_participants': Int64, 'num_adults': Int64, 'num_kids': Int64, 'num_workers': Int64, 'num_stu

## Write the filtered prepared tables

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_files = {}
for table_id, df in filtered_tables.items():
    path = output_path(OUTPUT_DIR, table_id, OUTPUT_FORMAT)
    write_table(df, path)
    output_files[table_id] = str(path.resolve())

print("Wrote filtered prepared tables to:")
for table_id, path in output_files.items():
    print(f"{table_id:25s} -> {path}")

Wrote filtered prepared tables to:
households                -> C:\Users\wesley.darling\projects\activitysim_visualizer\example_output_files\base_households_filtered\households.csv
persons                   -> C:\Users\wesley.darling\projects\activitysim_visualizer\example_output_files\base_households_filtered\persons.csv
tours                     -> C:\Users\wesley.darling\projects\activitysim_visualizer\example_output_files\base_households_filtered\tours.csv
trips                     -> C:\Users\wesley.darling\projects\activitysim_visualizer\example_output_files\base_households_filtered\trips.csv
joint_tour_participants   -> C:\Users\wesley.darling\projects\activitysim_visualizer\example_output_files\base_households_filtered\joint_tour_participants.csv
land_use                  -> C:\Users\wesley.darling\projects\activitysim_visualizer\example_output_files\base_households_filtered\land_use.csv


## Copy this into `prepared_table_map`

This block generates a YAML-ready mapping for your config.

In [10]:
print("prepared_table_map:")
for table_id in TABLE_IDS:
    normalized = output_files[table_id].replace("\\", "/")
    print(f"  {table_id}: {normalized}")

prepared_table_map:
  households: C:/Users/wesley.darling/projects/activitysim_visualizer/example_output_files/base_households_filtered/households.csv
  persons: C:/Users/wesley.darling/projects/activitysim_visualizer/example_output_files/base_households_filtered/persons.csv
  tours: C:/Users/wesley.darling/projects/activitysim_visualizer/example_output_files/base_households_filtered/tours.csv
  trips: C:/Users/wesley.darling/projects/activitysim_visualizer/example_output_files/base_households_filtered/trips.csv
  joint_tour_participants: C:/Users/wesley.darling/projects/activitysim_visualizer/example_output_files/base_households_filtered/joint_tour_participants.csv
  land_use: C:/Users/wesley.darling/projects/activitysim_visualizer/example_output_files/base_households_filtered/land_use.csv


In [11]:
config_example = {
    "runs": [
        {
            "label": "Filtered Prepared Example",
            "prepared_table_map": output_files,
        }
    ]
}

print(json.dumps(config_example, indent=2))

{
  "runs": [
    {
      "label": "Filtered Prepared Example",
      "prepared_table_map": {
        "households": "C:\\Users\\wesley.darling\\projects\\activitysim_visualizer\\example_output_files\\base_households_filtered\\households.csv",
        "persons": "C:\\Users\\wesley.darling\\projects\\activitysim_visualizer\\example_output_files\\base_households_filtered\\persons.csv",
        "tours": "C:\\Users\\wesley.darling\\projects\\activitysim_visualizer\\example_output_files\\base_households_filtered\\tours.csv",
        "trips": "C:\\Users\\wesley.darling\\projects\\activitysim_visualizer\\example_output_files\\base_households_filtered\\trips.csv",
        "joint_tour_participants": "C:\\Users\\wesley.darling\\projects\\activitysim_visualizer\\example_output_files\\base_households_filtered\\joint_tour_participants.csv",
        "land_use": "C:\\Users\\wesley.darling\\projects\\activitysim_visualizer\\example_output_files\\base_households_filtered\\land_use.csv"
      }
    }
  ]